# Descriptive Statistics — Exploring a PostgreSQL Database on Azure

This notebook shows how to:

1. **Connect to a PostgreSQL database hosted on Azure** (the same server used by the VS Code PostgreSQL extension),
2. **Pull data into pandas DataFrames** with SQL, and
3. **Explore it with the standard descriptive-statistics plots** — histogram, density, ECDF, bar chart, box plot, violin plot, scatter plot, time line, and correlation heatmap — with notes on *which question each plot answers*.

**The data.** The `classproject` database backs the class **project-matching platform** (a Django app). The tables we use:

| Table | Rows (Jul 2026) | Contents |
|---|---|---|
| `accounts_user` | 59 | registered students: department, major, languages, gender, birth year, study year, height |
| `projects_project` | 18 | proposed projects: title, target group size, status |
| `projects_application` | 34 | applications to join a project: status, timestamps |
| `projects_membership` | 30 | confirmed project members |

(`reports_report` and `reports_peerreview` exist too, but are still nearly empty — nothing to plot yet.)

**A note on privacy.** This is *real classmate data*. We deliberately never `SELECT` the `password`, `email`, or `full_name` columns, and every figure below is aggregated — no plot identifies an individual.


## 0. Setup

Install the required packages once (safe to re-run):

- `sqlalchemy` + `psycopg2-binary` — talk to PostgreSQL
- `pandas` — run SQL queries into DataFrames
- `matplotlib` + `seaborn` — plotting


In [ ]:
%pip install --quiet pandas sqlalchemy psycopg2-binary matplotlib seaborn

## 1. Connect to Azure PostgreSQL

The VS Code PostgreSQL extension and Python connect to the **same server with the same credentials** — the extension just stores them in a connection profile (VS Code → PostgreSQL sidebar → right-click the connection → *Edit Connection*):

| Setting | Value (from the `IssClassProject` profile) |
|---|---|
| Host | `classproject-iss-bfsu-db.postgres.database.azure.com` |
| Port | `5432` |
| Database | `classproject` |
| User | `classadmin` |
| SSL | **required** (Azure enforces `sslmode=require`) |

Two things that matter in practice:

- **Never hard-code the password in a notebook** — this file lives in a git repository. Below we read it from the `PGPASSWORD` environment variable if set, otherwise prompt for it with `getpass` (input stays hidden and is never saved in the file).
- **URL-encode the password** with `quote_plus`. Passwords often contain characters like `&`, `@` or `:` that break a plain connection URL.


In [ ]:
import os
from getpass import getpass
from urllib.parse import quote_plus

from sqlalchemy import create_engine, text
import pandas as pd

HOST = "classproject-iss-bfsu-db.postgres.database.azure.com"
PORT = 5432
DBNAME = "classproject"
USER = "classadmin"

# Prefer an environment variable; fall back to an interactive (hidden) prompt.
password = os.environ.get("PGPASSWORD") or getpass(f"Password for {USER}: ")

url = (
    f"postgresql+psycopg2://{USER}:{quote_plus(password)}@{HOST}:{PORT}/{DBNAME}"
    "?sslmode=require"          # Azure PostgreSQL requires SSL
)

# pool_pre_ping revalidates idle connections -- Azure closes them after a while
engine = create_engine(url, pool_pre_ping=True)

with engine.connect() as conn:
    print("Connected to:", conn.execute(text("SELECT version();")).scalar())

If the cell above prints a `PostgreSQL 16.x ...` version string, the connection works.

**Common errors and what they mean**

- `timeout expired` / `could not connect` — the Azure server's firewall does not allow your current IP. Add it with the Azure CLI (or Portal → server → *Networking*):
  ```
  az postgres flexible-server firewall-rule create \
      --resource-group classproject-rg --server-name classproject-iss-bfsu-db \
      --name allow-my-ip --start-ip-address <your.public.ip>
  ```
- `password authentication failed` — wrong password (watch for trailing spaces).
- `no pg_hba.conf entry ... no encryption` — you dropped `?sslmode=require` from the URL.

### 1.1 What tables are available?


In [ ]:
pd.read_sql(
    """
    SELECT table_name
    FROM information_schema.tables
    WHERE table_schema = 'public' AND table_type = 'BASE TABLE'
    ORDER BY table_name;
    """,
    engine,
)

The `django_*`, `auth_*` and `accounts_user_*` tables are framework plumbing — our data lives in `accounts_user`, `projects_*` and `reports_*`.

## 2. Load the data

`pd.read_sql(query, engine)` runs any SQL and returns a DataFrame. We select **only the columns we need** (and never the sensitive ones), and exclude the one staff/admin account so the user table contains just students.


In [ ]:
users = pd.read_sql(
    """
    SELECT id, department, major, languages, gender,
           birth_year, study_year, height_cm, date_joined
    FROM accounts_user
    WHERE is_staff = FALSE;      -- students only, drop the admin account
    """,
    engine,
)

projects = pd.read_sql(
    "SELECT id, title, group_size, status, created_at, owner_id FROM projects_project;",
    engine,
)

applications = pd.read_sql(
    """
    SELECT id, status, discussed_with_owner, created_at, decided_at,
           applicant_id, project_id
    FROM projects_application;
    """,
    engine,
)

memberships = pd.read_sql(
    "SELECT id, joined_at, member_id, project_id FROM projects_membership;",
    engine,
)

print(f"users {users.shape}, projects {projects.shape}, "
      f"applications {applications.shape}, memberships {memberships.shape}")
users.head(3)

### 2.1 Derived columns

A little feature engineering before plotting: age from birth year, a readable gender label, and per-project counts of applications and members (note the **grain**: `users` is one row per student, `projects` one row per project — always know which one you're plotting).


In [ ]:
THIS_YEAR = pd.Timestamp.now().year
users["age"] = THIS_YEAR - users["birth_year"]

# gender is 'F', 'M', plus a couple of blank/'N' entries -> group the rare ones
users["gender_label"] = (users["gender"].map({"F": "Female", "M": "Male"})
                                        .fillna("Other / unspecified"))

# Per-project counts, joined onto the projects table
projects["n_applications"] = (projects["id"]
                              .map(applications["project_id"].value_counts())
                              .fillna(0).astype(int))
projects["n_members"] = (projects["id"]
                         .map(memberships["project_id"].value_counts())
                         .fillna(0).astype(int))
projects["fill_rate"] = (projects["n_members"] / projects["group_size"]).round(2)

projects[["title", "group_size", "n_applications", "n_members", "fill_rate", "status"]].head()

## 3. Descriptive statistics in numbers

Always look at the numbers before plotting. `describe()` gives the five-number summary (plus mean and standard deviation); `value_counts()` gives categorical frequencies.


In [ ]:
users[["age", "study_year", "height_cm"]].describe().round(2)

In [ ]:
users["gender_label"].value_counts()

In [ ]:
# Grouped summary: the numeric backbone of the box plot in section 6
(users.groupby("gender_label")["height_cm"]
      .agg(["count", "mean", "median", "std", "min", "max"])
      .round(1))

### 3.1 A caution: free-text categories are messy

`department` and `major` were typed by hand, so the same department appears under many spellings and languages (e.g. `国际新闻与传播学院`, `Journalism`, `journalist`, `School of International Journalism and Communication` are arguably one group). **Don't group by a free-text column without cleaning it first** — the counts below are honest but nearly meaningless as categories:


In [ ]:
users["department"].value_counts().head(10)

Cleaning this properly means building a mapping table (raw value → canonical department) — a worthwhile exercise, but out of scope here. We stick to the *structured* columns (`gender`, `study_year`, `birth_year`, `height_cm`) for grouped plots.

## 4. Plot styling

One setup cell so every figure looks consistent: a colorblind-safe palette with one **fixed color per category** (the color follows the category across all plots, never its rank or position), muted grid and axis ink, no top/right spines.


In [ ]:
import matplotlib.pyplot as plt
from matplotlib.ticker import MaxNLocator
import seaborn as sns

# Ink & chrome
INK, INK2, MUTED = "#0b0b0b", "#52514e", "#898781"
GRID, SURFACE, BASELINE = "#e1e0d9", "#fcfcfb", "#c3c2b7"

# One accent hue for single-series plots
BLUE = "#2a78d6"

# Fixed colors for the gender groups, reused in every figure below
GENDER_ORDER = ["Female", "Male", "Other / unspecified"]
GENDER_COLORS = {"Female": "#2a78d6", "Male": "#1baf7a",
                 "Other / unspecified": "#898781"}

sns.set_theme(style="ticks", rc={
    "figure.facecolor": SURFACE, "axes.facecolor": SURFACE,
    "axes.edgecolor": BASELINE, "axes.labelcolor": INK2, "text.color": INK,
    "xtick.color": MUTED, "ytick.color": MUTED,
    "axes.grid": True, "grid.color": GRID, "grid.linewidth": 0.8,
    "axes.spines.top": False, "axes.spines.right": False,
    "figure.dpi": 110, "figure.figsize": (7.5, 4.2),
})

## 5. Distribution of one numeric variable

### 5.1 Histogram

*Question: "What values does this variable take, and how often?"*

The histogram bins a numeric variable and counts observations per bin. Sanity-check the **bin width**: too few bins hides structure, too many shows noise. With heights, 3 cm bins work well.


In [ ]:
fig, ax = plt.subplots()
ax.hist(users["height_cm"].dropna(), bins=range(155, 190, 3),
        color=BLUE, edgecolor=SURFACE, linewidth=1.5)

mean_h = users["height_cm"].mean()
ax.axvline(mean_h, color=INK2, linestyle="--", linewidth=1)
ax.annotate(f"mean = {mean_h:.1f} cm", xy=(mean_h, ax.get_ylim()[1] * 0.92),
            xytext=(6, 0), textcoords="offset points", color=INK2)

ax.set_xlabel("Height (cm)")
ax.set_ylabel("Number of students")
ax.set_title("Histogram — distribution of student heights", loc="left")
plt.show()

### 5.2 Density (KDE) plot

*Question: "What is the overall shape — and how does it differ between groups?"*

A kernel density estimate is a smoothed histogram, and it shines when **overlaying groups** where stacked histograms become unreadable. Here: height by gender (only the two groups with enough observations — a density of 1–2 points is pure smoothing artifact).


In [ ]:
fig, ax = plt.subplots()
for g in ["Female", "Male"]:
    subset = users.loc[users["gender_label"] == g, "height_cm"].dropna()
    sns.kdeplot(subset, ax=ax, color=GENDER_COLORS[g], linewidth=2,
                label=f"{g} (n={len(subset)})")

ax.set_xlabel("Height (cm)")
ax.set_ylabel("Density")
ax.set_title("Density — height by gender", loc="left")
ax.legend(frameon=False)
plt.show()

### 5.3 ECDF plot

*Question: "What share of observations is at or below a given value?"*

The empirical cumulative distribution function makes no binning or smoothing choices at all, and quantiles read straight off it — the y-value at age 20 is the share of students aged 20 or younger.


In [ ]:
fig, ax = plt.subplots()
sns.ecdfplot(users["age"], ax=ax, color=BLUE, linewidth=2)

median_age = users["age"].median()
ax.axhline(0.5, color=GRID, linewidth=1)
ax.axvline(median_age, color=INK2, linestyle="--", linewidth=1)
ax.annotate(f"median = {median_age:.0f}", xy=(median_age, 0.5),
            xytext=(8, -14), textcoords="offset points", color=INK2)

ax.set_xlabel("Age (years)")
ax.set_ylabel("Cumulative share of students")
ax.set_title("ECDF — student ages", loc="left")
plt.show()

## 6. Comparing categories

### 6.1 Bar chart — ordinal categories

*Question: "How large is each category?"*

Rules of thumb: start the axis at zero, label bars directly, and **sort by value — unless the categories have a natural order**. Study year is ordinal (1→4), so we keep its natural order instead of sorting by count.


In [ ]:
counts = users["study_year"].value_counts().sort_index()

fig, ax = plt.subplots()
bars = ax.bar(counts.index.astype(int).astype(str), counts.values,
              color=BLUE, width=0.62)
ax.bar_label(bars, padding=3, color=INK2)

ax.set_xlabel("Study year")
ax.set_ylabel("Number of students")
ax.set_title("Bar chart — students per study year", loc="left")
ax.grid(axis="x", visible=False)
plt.show()

### 6.2 Bar chart — nominal categories, sorted

Project titles have no natural order, so here we **do** sort by value, use horizontal bars (long labels stay readable), and label directly.


In [ ]:
top = (projects.assign(short_title=projects["title"].str.slice(0, 32))
               .sort_values("n_applications")
               .tail(10))                      # 10 most-applied-to projects

fig, ax = plt.subplots(figsize=(7.5, 4.6))
bars = ax.barh(top["short_title"], top["n_applications"], color=BLUE, height=0.62)
ax.bar_label(bars, padding=4, color=INK2)

ax.set_xlabel("Number of applications")
ax.set_title("Bar chart — applications per project (top 10)", loc="left")
ax.grid(axis="y", visible=False)
ax.xaxis.set_major_locator(MaxNLocator(integer=True))   # counts -> whole-number ticks
plt.show()

### 6.3 Box plot

*Question: "How does the **distribution** of a numeric variable differ across categories?"*

Each box shows median (line), interquartile range (box), whiskers, and outliers — the visual version of the grouped table in section 3. We keep only groups with enough data to summarize.


In [ ]:
two = users[users["gender_label"].isin(["Female", "Male"])]

fig, ax = plt.subplots()
sns.boxplot(data=two, x="gender_label", y="height_cm",
            order=["Female", "Male"], hue="gender_label",
            palette=GENDER_COLORS, legend=False,
            width=0.45, linewidth=1.2, fliersize=3, ax=ax)

ax.set_xlabel("")
ax.set_ylabel("Height (cm)")
ax.set_title("Box plot — height by gender", loc="left")
ax.grid(axis="x", visible=False)
plt.show()

### 6.4 Violin plot

Same question as the box plot, but also shows the **shape** of each distribution (skew, bimodality) that a box hides. Needs a reasonable group size to be meaningful.


In [ ]:
fig, ax = plt.subplots()
sns.violinplot(data=two, x="gender_label", y="height_cm",
               order=["Female", "Male"], hue="gender_label",
               palette=GENDER_COLORS, legend=False,
               inner="quart", linewidth=1, cut=0, ax=ax)

ax.set_xlabel("")
ax.set_ylabel("Height (cm)")
ax.set_title("Violin plot — height by gender", loc="left")
ax.grid(axis="x", visible=False)
plt.show()

## 7. Relationships and change over time

### 7.1 Scatter plot

*Question: "How do two numeric variables move together?"*

One point per **student**. Color encodes gender — the same fixed colors as every plot above. A little jitter on age (whole years) keeps overlapping points visible.


In [ ]:
import numpy as np

rng = np.random.default_rng(42)
plot_df = users.dropna(subset=["age", "height_cm"]).copy()
plot_df["age_jitter"] = plot_df["age"] + rng.uniform(-0.18, 0.18, len(plot_df))

fig, ax = plt.subplots()
for g in GENDER_ORDER:
    sub = plot_df[plot_df["gender_label"] == g]
    ax.scatter(sub["age_jitter"], sub["height_cm"], s=45,
               color=GENDER_COLORS[g], label=f"{g} (n={len(sub)})",
               edgecolor=SURFACE, linewidth=0.8)

ax.set_xlabel("Age (years, jittered)")
ax.set_ylabel("Height (cm)")
ax.set_title("Scatter — height vs. age, one point per student", loc="left")
ax.legend(frameon=False, bbox_to_anchor=(1.02, 1), loc="upper left")
plt.show()

### 7.2 Line / step plot over time

*Question: "How does a quantity change over time?"*

Every row carries a timestamp (`date_joined`), so we can plot **cumulative registrations**: sort by time, then count. The steep segments show when most of the class signed up.


In [ ]:
signups = users.sort_values("date_joined")

fig, ax = plt.subplots()
ax.step(signups["date_joined"], range(1, len(signups) + 1),
        where="post", color=BLUE, linewidth=2)

ax.set_xlabel("Date joined")
ax.set_ylabel("Registered students (cumulative)")
ax.set_title("Step line — platform registrations over time", loc="left")
fig.autofmt_xdate()
plt.show()

### 7.3 Correlation heatmap

*Question: "Which numeric variables are linearly related — across all pairs at once?"*

At the **project** grain: does asking for a bigger group attract more applications and end with more members? Correlations run from −1 to +1, so the color scale must be **diverging** (two hues meeting at neutral gray for 0), and cells are annotated because ±0.2 and ±0.5 look similar as color alone. With only 18 projects, treat these as descriptive, not conclusive.


In [ ]:
metrics = projects[["group_size", "n_applications", "n_members"]]
corr = metrics.corr()

fig, ax = plt.subplots(figsize=(5.4, 4.2))
sns.heatmap(corr, annot=True, fmt=".2f", center=0, vmin=-1, vmax=1,
            cmap=sns.diverging_palette(255, 12, as_cmap=True),
            linewidths=2, linecolor=SURFACE,
            cbar_kws={"label": "Pearson r"}, ax=ax)
ax.set_title("Correlation heatmap — project-level metrics (n=18)", loc="left")
plt.show()

### 7.4 Pair plot

The scatter-plot matrix: every numeric variable against every other, distributions on the diagonal. Use it as a **first look** at a new dataset, not as a final figure — it is too dense to communicate one message.


In [ ]:
grid = sns.pairplot(two[["age", "height_cm", "study_year", "gender_label"]].dropna(),
                    hue="gender_label", palette=GENDER_COLORS,
                    hue_order=["Female", "Male"],
                    height=2.1, corner=True,
                    plot_kws={"s": 32, "edgecolor": SURFACE, "linewidth": 0.6})
grid.figure.suptitle("Pair plot — student-level variables", x=0.38, y=1.02)
plt.show()

## 8. Which plot for which question?

| You want to show… | Use | Section |
|---|---|---|
| The distribution of one numeric variable | Histogram / KDE / ECDF | 5.1–5.3 |
| Sizes of categories | Bar chart (zero-based, labeled; sort nominal, keep ordinal order) | 6.1–6.2 |
| A numeric distribution *per category* | Box plot (summary) / violin (shape) | 6.3–6.4 |
| The relationship between two numeric variables | Scatter plot | 7.1 |
| Change over time | Line / step plot | 7.2 |
| All pairwise linear relationships | Correlation heatmap / pair plot | 7.3–7.4 |

Habits worth keeping:

- Know the **grain** of each table (per student vs. per project) before plotting.
- Keep **one fixed color per category** across all figures.
- **Free-text columns are not categories** until cleaned (section 3.1).
- Drop or merge groups too small to summarize; say so in the caption.
- Diverging color scales only for signed quantities (like correlations).
- With small n, plots describe *this class*, they don't estimate a population.

Finally, release the database connections held by this notebook:


In [ ]:
engine.dispose()